In [11]:
import pandas as pd
from collections import Counter
import os

In [12]:
bank_files = [
  'BCAMobile_reviews_cleaned.csv', 
  'BRImo_reviews_cleaned.csv', 
  'livin_mandiri_reviews_cleaned.csv', 
  'Wondr_BNI_reviews_cleaned.csv'
]

semua_kandidat = []

In [13]:
for file in bank_files:
  path = f'data/processed/{file}'
  if os.path.exists(path):
    # 1. Load data bersih per bank
    df = pd.read_csv(path)
    df = df.dropna(subset=['content_cleaned'])
    
    # 2. Tokenisasi kata untuk bank ini saja
    teks_bank = " ".join(df['content_cleaned'].astype(str))
    kata_bank = teks_bank.split()
    
    # 3. Ambil Top 500 kata paling sering muncul di bank ini
    frekuensi = Counter(kata_bank)
    top_500 = frekuensi.most_common(500)
    
    # 4. Masukkan ke dalam list kandidat beserta keterangan sumber bank-nya
    for kata, jumlah in top_500:
      semua_kandidat.append({
          'Kata Asli': kata,
          'Frekuensi': jumlah,
          'Sumber': file.split('_')[0] # Ambil nama depannya aja (BCAMobile, BRImo, dll)
      })

In [14]:
df_kandidat = pd.DataFrame(semua_kandidat)

# 6. HAPUS DUPLIKAT & SIAPKAN KOLOM KOSONG
df_unik = df_kandidat.drop_duplicates(subset=['Kata Asli'], keep='first').copy()
df_unik['Kata Baku (Isi Manual)'] = ""

In [15]:
df_unik.head(10)

,Kata Asli,Frekuensi,Sumber,Kata Baku (Isi Manual)
0,bca,11626,BCAMobile,
1,nya,7670,BCAMobile,
2,ga,6323,BCAMobile,
3,aplikasi,6130,BCAMobile,
4,gak,5782,BCAMobile,
5,mau,5763,BCAMobile,
6,terus,5741,BCAMobile,
7,update,5708,BCAMobile,
8,verifikasi,5502,BCAMobile,
9,malah,5139,BCAMobile,


In [16]:
import requests
import io

slang_word_dict = 'https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv'

try:
  response = requests.get(slang_word_dict)
  df_alay = pd.read_csv(io.StringIO(response.text))
  # Buat dictionary (kata_gaul -> kata_baku)
  dict_alay = dict(zip(df_alay['slang'], df_alay['formal']))
  print("   -> Kamus berhasil diunduh dan siap digunakan!")
except Exception as e:
  print(f"   -> Gagal mengunduh kamus: {e}")
  dict_alay = {}

   -> Kamus berhasil diunduh dan siap digunakan!


In [17]:
def auto_fill_abbreviation(kata):
  # Jika kata ada di kamus GitHub, kembalikan kata bakunya. Jika tidak, kosongkan.
  return dict_alay.get(kata, "")

df_unik['Kata Baku'] = df_unik['Kata Asli'].apply(auto_fill_abbreviation)
terisi = df_unik[df_unik['Kata Baku'] != ""]
kosong = df_unik[df_unik['Kata Baku'] == ""]

print(f"Berhasil diisi otomatis : {len(terisi)} kata")
print(f"Masih kosong            : {len(kosong)} kata")

Berhasil diisi otomatis : 103 kata
Masih kosong            : 569 kata


In [18]:
output_filename = 'data/slang_words_combined.csv'
df_unik.to_csv(output_filename, index=False)